<a href="https://colab.research.google.com/github/Sathwik828/renifrocment-Learinig-Lab/blob/main/Rl10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Practical 10: Bellman Optimality and Value Iteration
# Deterministic 2 x 2 Grid World

# 1. Define states, actions, and parameters
states = ["S0", "S1", "G"]
actions = ["up", "down", "left", "right"]

gamma = 0.9
theta = 1e-6

# State values are initialized to zero.
V = {
    "S0": 0.0,
    "S1": 0.0,
    "G": 0.0
}

# Grid layout:
#   S0 | WALL
#   S1 | G

# 2. Define deterministic transitions.
# If the agent hits a wall or boundary, it remains
# in its current state and receives reward -1.

def get_transition(state, action):
    if state == "G":
        return "G", 0

    transitions = {
        "S0": {
            "up":    ("S0", -1),
            "down":  ("S1", -1),
            "left":  ("S0", -1),
            "right": ("S0", -1)  # Wall
        },
        "S1": {
            "up":    ("S0", -1),
            "down":  ("S1", -1),
            "left":  ("S1", -1),
            "right": ("G", 10)
        }
    }

    return transitions[state][action]


# 3. Inspect the MDP
print("STATES:", states)
print("ACTIONS:", actions)
print("Discount factor gamma:", gamma)
print("Normal-cell reward: -1")
print("Wall/boundary reward: -1")
print("Goal reward: +10")
print("\nTRANSITION MAPPING")

for state in ["S0", "S1"]:
    for action in actions:
        next_state, reward = get_transition(state, action)
        print(
            f"State: {state:2s} | Action: {action:5s} "
            f"| Next state: {next_state:2s} "
            f"| Reward: {reward}"
        )


# 4. Verify all state-action transitions
assert all(
    get_transition(s, a) is not None
    for s in ["S0", "S1"]
    for a in actions
)
print("\nAll non-terminal state-action transitions are defined.")


# 5. Print initial state values
print("\nINITIAL STATE VALUES")
for state, value in V.items():
    print(f"V({state}) = {value:.6f}")


# 6. Implement synchronous Value Iteration
# Each iteration uses only the previous iteration's values.

iteration = 0

print("\nVALUE ITERATION")
print(f"{'Iteration':>9} | {'V(S0)':>12} | "
      f"{'V(S1)':>12} | {'V(G)':>12} | {'Delta':>12}")

while True:
    iteration += 1

    # Copy previous values; G remains terminal.
    old_V = V.copy()
    new_V = {"G": 0.0}

    # Calculate every new value using old_V only.
    for state in ["S0", "S1"]:
        q_values = []

        for action in actions:
            next_state, reward = get_transition(state, action)
            q = reward + gamma * old_V[next_state]
            q_values.append(q)

        new_V[state] = max(q_values)

    # Maximum change in state values
    delta = max(
        abs(new_V[s] - old_V[s])
        for s in states
    )

    V = new_V

    print(
        f"{iteration:9d} | {V['S0']:12.6f} | "
        f"{V['S1']:12.6f} | {V['G']:12.6f} | "
        f"{delta:12.6f}"
    )

    if delta < theta:
        break


# 7. Calculate final action values Q*(s, a)
print("\nFINAL OPTIMAL STATE VALUES")
for state in states:
    print(f"V*({state}) = {V[state]:.6f}")


print("\nFINAL ACTION VALUES Q*(s, a)")

Q = {}

for state in ["S0", "S1"]:
    Q[state] = {}

    for action in actions:
        next_state, reward = get_transition(state, action)
        q = reward + gamma * V[next_state]
        Q[state][action] = q

        print(
            f"Q*({state}, {action}) = {q:.6f}"
        )


# 8. Extract the optimal policy.
# Report all actions in case of ties.

print("\nOPTIMAL POLICY")

policy = {}

for state in ["S0", "S1"]:
    best_value = max(Q[state].values())

    best_actions = [
        action
        for action, value in Q[state].items()
        if abs(value - best_value) < 1e-9
    ]

    policy[state] = best_actions

    print(
        f"State {state}: {best_actions} "
        f"(Q-value = {best_value:.6f})"
    )


# 9. Verify Bellman Optimality numerically.

print("\nBELLMAN OPTIMALITY VERIFICATION")

for state in ["S0", "S1"]:
    max_q = max(Q[state].values())

    print(
        f"State {state}: V* = {V[state]:.6f}, "
        f"max Q* = {max_q:.6f}"
    )

    assert abs(V[state] - max_q) < 1e-9

print("\nVerification successful!")
print("V*(s) = max_a Q*(s, a) for every non-terminal state.")


# 10. Explain the result

print("\nRESULT EXPLANATION")
print("1. S0: Move DOWN to reach S1.")
print("2. S1: Move RIGHT to reach the goal G.")
print("3. The goal reward is +10, increasing the value of S1.")
print("4. Gamma = 0.9 discounts future rewards.")
print("5. The optimal path is S0 -> S1 -> G.")


STATES: ['S0', 'S1', 'G']
ACTIONS: ['up', 'down', 'left', 'right']
Discount factor gamma: 0.9
Normal-cell reward: -1
Wall/boundary reward: -1
Goal reward: +10

TRANSITION MAPPING
State: S0 | Action: up    | Next state: S0 | Reward: -1
State: S0 | Action: down  | Next state: S1 | Reward: -1
State: S0 | Action: left  | Next state: S0 | Reward: -1
State: S0 | Action: right | Next state: S0 | Reward: -1
State: S1 | Action: up    | Next state: S0 | Reward: -1
State: S1 | Action: down  | Next state: S1 | Reward: -1
State: S1 | Action: left  | Next state: S1 | Reward: -1
State: S1 | Action: right | Next state: G  | Reward: 10

All non-terminal state-action transitions are defined.

INITIAL STATE VALUES
V(S0) = 0.000000
V(S1) = 0.000000
V(G) = 0.000000

VALUE ITERATION
Iteration |        V(S0) |        V(S1) |         V(G) |        Delta
        1 |    -1.000000 |    10.000000 |     0.000000 |    10.000000
        2 |     8.000000 |    10.000000 |     0.000000 |     9.000000
        3 |     8.